In [ ]:
# task1_load_explore.py

import pandas as pd

# Load the dataset
df = pd.read_csv("churnguard_data.csv")

# Print shape of dataset
print("Shape of dataset:", df.shape)

# Print first 5 rows
print("\nLast 10 Rows:")
print(df.tail(10))

# Print column names and data types
print("\nDataset Info:")
print(df.info())

# Print missing values count
print("\nMissing Values in Each Column:")
print(df.isnull().sum())

# Print number of duplicate rows
print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())

# Print value counts of Churn column
print("\nValue Counts of Churn Column:")
print(df["Churn"].map({"Yes": 1, "No": 0}))


# Print unique values in Contract column
print("\nUnique Values in Contract Column:")
print(df["Contract"].unique())

Shape of dataset: (1030, 12)

Last 10 Rows:
     customerID      gender  SeniorCitizen  tenure PhoneService  \
1020  CUST-0309      Female              0    45.0          Yes   
1021  CUST-0662      Female              0    21.0          Yes   
1022  CUST-0131        Male              0    70.0          Yes   
1023  CUST-0664        Male              0    -1.0          Yes   
1024  CUST-0872        Male              0    -3.0          Yes   
1025  CUST-0088      Female              0    26.0          Yes   
1026  CUST-0331        Male              0    49.0          Yes   
1027  CUST-0467        Male              1    69.0          Yes   
1028  CUST-0122      Female              1     3.0          Yes   
1029  CUST-0861    Female                0    49.0          Yes   

     InternetService        Contract PaperlessBilling         PaymentMethod  \
1020     Fiber optic  Month-to-month               No      Electronic check   
1021             DSL        Two year               No      E

In [ ]:
#clean  the data:
import pandas as pd
df = pd.read_csv("churnguard_data.csv")
df = df.drop("customerID", axis=1)
df = df.drop_duplicates()
df["gender"] = df["gender"].str.strip()
df["paymentMethod"] = df["PaymentMethod"].str.strip()
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()
df["Contract"] = df["Contract"].str.strip().str.lower()
contract_mapping = {
    "monthly": "Month-to-Month",
    "month to month": "Month-to-Month",
    "1 year": "One year",
    "two year": "Two year",
    "2 year": "Two year"
}

df["Contract"] = df["Contract"].replace(contract_mapping)
df["InternetService"] = df["InternetService"].str.strip().str.lower()

internet_mapping = {
    "dsl": "DSL",
    "fibre optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "none":"No",
    "no":"No"
}
df["InternetService"] = df["InternetService"].replace(internet_mapping)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
df = df[df["tenure"] > 0]
df = df[(df["MonthlyCharges"] >= 10) & (df["MonthlyCharges"] <= 200)]
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())

df["tenure"] = df["tenure"].fillna(round(df["tenure"].median()))


print("Shape of Cleaned DataFrame:")
print(df.shape)


print("\nMissing Values After Cleaning:")
print(df.isnull().sum())



 




Shape of Cleaned DataFrame:
(867, 12)

Missing Values After Cleaning:
gender               0
SeniorCitizen        0
tenure               0
PhoneService         0
InternetService     14
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
paymentMethod        0
dtype: int64


In [ ]:
# task3_train_model.py

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("churnguard_data.csv")


# DATA CLEANING (Task 2 Steps)


# Drop customerID column
df = df.drop("customerID", axis=1)

# Remove duplicate rows
df = df.drop_duplicates()

# Strip whitespace
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# Standardize casing
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

# Fix Contract column
df["Contract"] = df["Contract"].str.strip().str.lower()

contract_mapping = {
    "monthly": "Month-to-month",
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",
    "1 year": "One year",
    "one year": "One year",
    "2 year": "Two year",
    "two year": "Two year"
}

df["Contract"] = df["Contract"].replace(contract_mapping)

# Fix InternetService column
df["InternetService"] = df["InternetService"].str.strip().str.lower()

internet_mapping = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "none": "No",
    "no": "No"
}

df["InternetService"] = df["InternetService"].replace(internet_mapping)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove invalid tenure values
df = df[df["tenure"] > 0]

# Remove outliers in MonthlyCharges
df = df[(df["MonthlyCharges"] >= 10) & (df["MonthlyCharges"] <= 200)]

# Fill missing values
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())

df["tenure"] = df["tenure"].fillna(round(df["tenure"].median()))


# ENCODING


# Encode target column
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# One-hot encoding
df = pd.get_dummies(
    df,
    columns=[
        "gender",
        "PhoneService",
        "InternetService",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod"
    ],
    drop_first=True
)


# SPLIT FEATURES AND TARGET


X = df.drop("Churn", axis=1)
y = df["Churn"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# TRAIN MODEL


model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# -----------------------------
# EVALUATION
# -----------------------------

# Accuracy score
print("Accuracy Score:")
print(accuracy_score(y_test, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stay", "Churn"]
))

Accuracy Score:
0.6954022988505747

Classification Report:
              precision    recall  f1-score   support

        Stay       0.75      0.85      0.80       122
       Churn       0.49      0.33      0.39        52

    accuracy                           0.70       174
   macro avg       0.62      0.59      0.59       174
weighted avg       0.67      0.70      0.68       174



C:\Users\kasir\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
df = pd.read_csv("churnguard_data.csv")

# Drop customerID column
df = df.drop("customerID", axis=1)

# Remove duplicate rows
df = df.drop_duplicates()

# Strip whitespace
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# Standardize casing
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

# Fix Contract column
df["Contract"] = df["Contract"].str.strip().str.lower()

contract_mapping = {
    "monthly": 0,
    "month to month": 0,
    "month-to-month": 0,
    "1 year": 1,
    "one year": 1,
    "2 year": 2,
    "two year": 2
}

df["Contract"] = df["Contract"].replace(contract_mapping)

# Fix InternetService column
df["InternetService"] = df["InternetService"].str.strip().str.lower()

internet_mapping = {
    "dsl": "DSL",
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "none": "No",
    "no": "No"
}

df["InternetService"] = df["InternetService"].replace(internet_mapping)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove invalid tenure values
df = df[df["tenure"] > 0]

# Remove outliers in MonthlyCharges
df = df[(df["MonthlyCharges"] >= 10) & (df["MonthlyCharges"] <= 200)]

# Fill missing values
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())

df["tenure"] = df["tenure"].fillna(round(df["tenure"].median()))

# Encode Churn column
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})


# SELECT FEATURES


X = df[[
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Contract"
]]

y = df["Churn"]


# TRAIN MODEL


model = LogisticRegression(max_iter=1000)

model.fit(X, y)


# USER INPUT


tenure = int(input("Enter tenure (months): "))

monthly_charges = float(input("Enter Monthly Charges: "))

total_charges = float(input("Enter Total Charges: "))

senior_citizen = int(input("Senior Citizen? (1 = Yes, 0 = No): "))

contract = int(input(
    "Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): "
))

# Create input data
new_customer = [[
    tenure,
    monthly_charges,
    total_charges,
    senior_citizen,
    contract
]]

# Predict churn
prediction = model.predict(new_customer)

# Print result
if prediction[0] == 1:
    print("Prediction: This customer is likely to CHURN.")
else:
    print("Prediction: This customer is likely to STAY.")

C:\Users\kasir\AppData\Local\Temp\ipykernel_14824\3508092914.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Contract"] = df["Contract"].replace(contract_mapping)


Enter tenure (months):  67
Enter Monthly Charges:  99
Enter Total Charges:  99
Senior Citizen? (1 = Yes, 0 = No):  90
Contract type (0 = Month-to-month, 1 = One year, 2 = Two year):  89


Prediction: This customer is likely to STAY.


C:\Users\kasir\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
